# Create Rashomon Set — ResNet50 with Different Weight Initializations

This notebook retrains ResNet50 classifiers on DermaMNIST with different random seeds
to build a Rashomon set of near-equivalent models.

**Changes from the original version (aligned with `train_classifier_derma_optimized.ipynb`):**

- **ImageNet normalization**: `A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))` instead of `mean=0.0, std=1.0`
- **Class weights**: proper inverse-frequency formula `total / (num_classes * count)` instead of `num_classes * (1 - count/total)`
- **Class weights injection**: passed via `CrossEntropyLoss(weight=...)` at construction, not assigned to a non-existent `.weights` attribute after the fact
- **Evaluation**: uses a proper `evaluate()` helper that handles device placement, instead of `evaluate_classification_model` which lacks a `device` parameter
- **LR scheduler monitor**: watches `val_loss` instead of `train_loss`
- **`num_classes`**: always derived from config, never recomputed from raw label counts

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import os
import copy
import torch
import numpy as np
import os.path as osp
from tqdm import tqdm
from random import randint

import albumentations as A
from torchmetrics import Accuracy

from lightning.pytorch import Trainer
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

from src.datasets import DatasetBuilder
from src.datasets.augmentations import AUGMENTATIONS
from src.datasets import datasets_api
from src.models.classifiers import build_resnet50
from src.models.lightning_wrappers import ClassifierLightningWrapper
from src.utils.generic_utils import seed_everything, get_config, load_model_weights

I0000 00:00:1775217765.876976    2698 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
seed_everything()

## 1. Config, ImageNet normalization & data loading

In [4]:
# ── ImageNet normalization (same as train_classifier_derma_optimized.ipynb) ──
# IMAGENET_MEAN = (0.485, 0.456, 0.406)
# IMAGENET_STD  = (0.229, 0.224, 0.225)

In [3]:
repo_path = '/teamspace/studios/this_studio/CF-Robustness-Benchmark'
config_path = osp.join(repo_path, 'configs/train_classifier_derma.yaml')
# config_path = osp.join(repo_path, 'configs/train_classifier_derma_optimized.yaml')
config = get_config(config_path)

ckpt_path = '/teamspace/studios/this_studio/CF-Robustness-Benchmark/notebooks/experiments/dermamnist_classification/binary/checkpoints/derma_resnet50_acc=0.79.pth'
# ckpt_path = '/teamspace/studios/this_studio/CF-Robustness-Benchmark/notebooks/experiments/dermamnist_classification/multiclass/checkpoints/derma_resnet50_acc=0.74.pth'
# ckpt_path = 'notebooks/experiments/dermamnist_classification/multiclass/checkpoints/derma_resnet50_acc=0.74_fold_3.pth'
# ckpt_path = 'notebooks/experiments/dermamnist_classification/binary/checkpoints/derma_resnet50_acc=0.72.pth'
config.classifier.checkpoints_path = osp.join(
    repo_path, ckpt_path)
config.data_dir = osp.join(repo_path, 'data')

# Derived constants (single source of truth)
NUM_CLASSES = config.data.num_classes
DEVICE = config.accelerator if torch.cuda.is_available() else 'cpu'
TASK = 'binary' if NUM_CLASSES == 2 else 'multiclass'

print(f"Device: {DEVICE} | Classes: {NUM_CLASSES} | Task: {TASK}")

Device: cuda | Classes: 2 | Task: binary


In [4]:
ds_builder = DatasetBuilder(config)
ds_builder.setup()
train_loader, val_loader, test_loader = ds_builder.get_dataloaders()

imgs = ds_builder.train_dataset.data.imgs  # (N, H, W, 3), uint8 [0, 255]
mean = imgs.mean(axis=(0, 1, 2)) / 255.0   # per-channel mean
std = imgs.std(axis=(0, 1, 2)) / 255.0  

train_transform = A.Compose([
    # A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    # A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.3),
    A.Normalize(mean=mean.tolist(), std=std.tolist()),
])

train_loader.dataset.transform = train_transform
val_loader.dataset.transform = A.Compose(
            [A.Normalize(mean=mean.tolist(), std=std.tolist())]) 

test_loader.dataset.transform = A.Compose(
            [A.Normalize(mean=mean.tolist(), std=std.tolist())])

print(f"Train: {len(ds_builder.train_dataset)} | Val: {len(ds_builder.val_dataset)} | Test: {len(ds_builder.test_dataset)}")

Train: 1548 | Val: 221 | Test: 443


In [5]:
from torch.utils.data import WeightedRandomSampler
from torch.utils.data import DataLoader, Subset

def make_balanced_sampler(labels, num_classes):
    """
    Build a sampler that draws each class with equal probability.
    
    Args:
        labels: 1D array of integer class labels for every sample in the dataset.
        num_classes: total number of classes.
    
    Returns:
        WeightedRandomSampler instance (pass to DataLoader's `sampler` arg).
    """
    class_counts = np.bincount(labels, minlength=num_classes)
    
    # Weight per sample = 1 / (count of that sample's class)
    # So if melanocytic nevi has 4528 samples, each one gets weight 1/4528
    # And if basal cell carcinoma has 239 samples, each gets weight 1/239
    # This makes the expected number of draws per class equal across one epoch
    sample_weights = 1.0 / class_counts[labels]
    sample_weights = torch.from_numpy(sample_weights).double()
    
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(labels),  # epoch length stays the same
        replacement=True,         # minority samples get drawn multiple times
    )

train_labels = ds_builder.train_dataset.data.labels.ravel()
balanced_sampler = make_balanced_sampler(train_labels, NUM_CLASSES)

# Replace shuffle=True with sampler — they're mutually exclusive
train_loader = DataLoader(
    ds_builder.train_dataset,
    batch_size=config.batch_size,
    sampler=balanced_sampler,
)

train_loader.dataset.transform = train_transform

## 2. Helper functions

In [6]:
def compute_class_weights(labels, num_classes, device='cpu', smoothing='sqrt'):
    # labels = dataset.data.labels.ravel()
    class_counts = np.bincount(labels, minlength=num_classes)
    total = labels.shape[0]

    raw_weights = total / (num_classes * class_counts)
    
    if smoothing == 'sqrt':
        raw_weights = np.sqrt(raw_weights)
    elif smoothing == 'log':
        raw_weights = np.log1p(raw_weights)
    
    # Normalize so weights sum to num_classes (preserves loss scale)
    weights = raw_weights * num_classes / raw_weights.sum()
    weights = torch.tensor(weights, dtype=torch.float32).to(device)

    print("Class distribution:")
    for i, (count, w) in enumerate(zip(class_counts, weights)):
        print(f"  Class {i}: {count:>5d} samples  |  weight = {w:.4f}")
    return weights


@torch.no_grad()
def evaluate(model, dataloader, device):
    """Evaluate model accuracy on dataloader. Returns accuracy as float."""
    model.eval()
    metric = Accuracy(task=TASK, num_classes=NUM_CLASSES).to(device)
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        preds = torch.argmax(model(images), dim=1)
        metric.update(preds, labels)
    return metric.compute().item()

## 3. Evaluate baseline classifier

In [7]:
baseline_classifier = build_resnet50(NUM_CLASSES, 
                                    pretrained=config.classifier.args.pretrained,
                                    unfreeze_last_n=config.classifier.args.unfreeze_last_n)
load_model_weights(
    baseline_classifier,
    weights_path=config.classifier.checkpoints_path,
    lightning_used=False,
)
baseline_classifier = baseline_classifier.to(DEVICE)

baseline_accuracy = evaluate(baseline_classifier, test_loader, DEVICE)
print(f"Baseline test accuracy: {baseline_accuracy:.3%}")

Baseline test accuracy: 79.007%


## 4. Create Rashomon set

In [9]:
# ── Directory setup ──
expt_dir = osp.join(repo_path, 'notebooks/experiments')
expt_name = f'{config.data.name}_classification'
expt_version = 'binary' if NUM_CLASSES == 2 else 'multiclass'
checkpoints_dir = osp.join(expt_dir, expt_name, expt_version, 'checkpoints', 'mc_2_4')
os.makedirs(checkpoints_dir, exist_ok=True)

class_names = ds_builder.class_encodings
classes4fname = '_'.join(str(v) for v in class_names.values()) if NUM_CLASSES == 2 else ''

# ── Logger ──
tb_logger = TensorBoardLogger(save_dir=expt_dir, name=expt_name, version=expt_version)

In [10]:
ACCURACY_TOLERANCE = 0.08  # max acceptable drop from baseline
N_MODELS = 10
MAX_EPOCHS = 40

seed_list = [randint(1000, 3000) for _ in range(N_MODELS)]
saved_models = []

for seed in tqdm(seed_list, desc='Rashomon set'):
    seed_everything(seed)

    # Build a fresh ResNet50 with new random head initialization
    cnn_wi = build_resnet50(NUM_CLASSES, 
                            pretrained=config.classifier.args.pretrained,
                            unfreeze_last_n=config.classifier.args.unfreeze_last_n)
    cnn_wrapper = ClassifierLightningWrapper(config, cnn_wi)

    # FIX: replace the loss with one that has proper class weights
    # cnn_wrapper.loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    # cnn_wrapper._custom_optimizer = torch.optim.Adam([
    #     {'params': cnn_wi.fc.parameters(), 'lr': 1e-3},
    #     {'params': [p for n, p in cnn_wi.named_parameters()
    #                 if 'fc' not in n and p.requires_grad], 'lr': 1e-5},
    # ])
    
    early_stop_callback = EarlyStopping(
        monitor='val_loss',
        min_delta=0.00,
        patience=10,
        verbose=False,
        mode='min',
    )

    trainer = Trainer(
        log_every_n_steps=10,
        max_epochs=MAX_EPOCHS,
        enable_checkpointing=False,
        callbacks=[early_stop_callback],
        logger=tb_logger,
    )
    trainer.fit(
        model=cnn_wrapper,
        train_dataloaders=train_loader,
        val_dataloaders=val_loader,
    )

    # Evaluate on test set
    cnn_wi = cnn_wi.to(DEVICE)
    accuracy = evaluate(cnn_wi, test_loader, DEVICE)
    print(f"  Seed {seed} | Test accuracy: {accuracy:.3%}")

    # Keep models within tolerance of the baseline
    if baseline_accuracy - accuracy < ACCURACY_TOLERANCE:
        fname = f'{config.data.name}_{classes4fname}_{seed}.pth'
        save_path = osp.join(checkpoints_dir, fname)
        torch.save(cnn_wi.state_dict(), save_path)
        saved_models.append((seed, accuracy, save_path))
        print(f"    -> Saved: {fname}")
    else:
        # Accuracy too low — add another seed to try
        seed_list.append(randint(1000, 3000))
        print(f"    -> Rejected (drop = {baseline_accuracy - accuracy:.3%}), adding new seed")

print(f"\nRashomon set complete: {len(saved_models)}/{len(seed_list)} models saved.")

Rashomon set:   0%|          | 0/10 [00:00<?, ?it/s]GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | model         | ResNet           | 23.8 M | train
1 | loss_fn       | CrossEntropyLoss | 0      | train
2 | train_metrics | MetricCollection | 0      | train
3 | valid_metrics | MetricCollection | 0      | train
-----------------------------------------------------------
23.8 M    Trainable params
0         Non-trainable params
23.8 M    Total params
95.082    Total estimated model params size (MB)
162       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Rashomon set: 100%|██████████| 10/10 [30:11<00:00, 181.18s/it]

  Seed 1760 | Test accuracy: 78.555%
    -> Saved: dermamnist_benign keratosis-like lesions_melanoma_1760.pth

Rashomon set complete: 10/10 models saved.


In [11]:
# Summary of saved models
print(f"Baseline accuracy: {baseline_accuracy:.3%}")
print(f"Tolerance: {ACCURACY_TOLERANCE:.0%}")
print(f"\nSaved models:")
for seed, acc, path in saved_models:
    delta = baseline_accuracy - acc 
    print(f"  seed={seed:>5d} | acc={acc:.3%} | delta={delta:.3f} | {osp.basename(path)}")

if saved_models:
    accs = [a for _, a, _ in saved_models]
    print(f"\nAccuracy range: [{min(accs):.3%}, {max(accs):.3%}]")
    print(f"Mean: {np.mean(accs):.3%} +/- {np.std(accs):.3%}")

Baseline accuracy: 79.007%
Tolerance: 8%

Saved models:
  seed= 2142 | acc=78.781% | delta=0.002 | dermamnist_benign keratosis-like lesions_melanoma_2142.pth
  seed= 1169 | acc=77.427% | delta=0.016 | dermamnist_benign keratosis-like lesions_melanoma_1169.pth
  seed= 2322 | acc=79.007% | delta=0.000 | dermamnist_benign keratosis-like lesions_melanoma_2322.pth
  seed= 2715 | acc=79.684% | delta=-0.007 | dermamnist_benign keratosis-like lesions_melanoma_2715.pth
  seed= 1978 | acc=79.007% | delta=0.000 | dermamnist_benign keratosis-like lesions_melanoma_1978.pth
  seed= 2939 | acc=77.878% | delta=0.011 | dermamnist_benign keratosis-like lesions_melanoma_2939.pth
  seed= 1354 | acc=78.330% | delta=0.007 | dermamnist_benign keratosis-like lesions_melanoma_1354.pth
  seed= 2083 | acc=78.330% | delta=0.007 | dermamnist_benign keratosis-like lesions_melanoma_2083.pth
  seed= 1000 | acc=77.652% | delta=0.014 | dermamnist_benign keratosis-like lesions_melanoma_1000.pth
  seed= 1760 | acc=78.555